# 🏗️ TASK 7: Production ML Pipelines
## From Messy Notebooks to Clean, Reusable Code
### Professional Pipeline Architecture with Scikit-Learn

---

## SETUP: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ All libraries imported')

✅ All libraries imported


---

## STEP 1: Load Titanic Dataset

In [ ]:
from google.colab import files

print('Upload titanic.csv:')
uploaded = files.upload()

df = pd.read_csv('titanic.csv')

print('\n' + '='*80)
print('DATASET LOADED')
print('='*80)
print(f'Shape: {df.shape}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nData types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())

Upload titanic.csv:


Saving titanic.csv to titanic.csv

DATASET LOADED
Shape: (891, 12)

First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803

---

## STEP 2: Feature Engineering - Create New Features

In [ ]:
print('\n' + '='*80)
print('FEATURE ENGINEERING')
print('='*80)

df_engineered = df.copy()

print('\n🔧 ENGINEERING NEW FEATURES:')

# Feature 1: Family Size
print('\n1. FamilySize = SibSp + Parch + 1')
df_engineered['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print(f'   Created. Range: {df_engineered["FamilySize"].min()} to {df_engineered["FamilySize"].max()}')
print(f'   Correlation with Survival: {df_engineered[["FamilySize", "Survived"]].corr().iloc[0, 1]:.4f}')

# Feature 2: Title from Name
print('\n2. Title = Extracted from Name (Mr, Mrs, Miss, etc)')
df_engineered['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
# Simplify rare titles
title_mapping = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'}
df_engineered['Title'] = df_engineered['Title'].map(title_mapping)
df_engineered['Title'].fillna('Other', inplace=True)
print(f'   Title distribution:')
print(df_engineered['Title'].value_counts())

# Feature 3: Age per Family Member
print('\n3. IsAlone = (FamilySize == 1)')
df_engineered['IsAlone'] = (df_engineered['FamilySize'] == 1).astype(int)
print(f'   Alone: {(df_engineered["IsAlone"]==1).sum()}, Not Alone: {(df_engineered["IsAlone"]==0).sum()}')

print('\n✅ Feature engineering complete!')


FEATURE ENGINEERING

🔧 ENGINEERING NEW FEATURES:

1. FamilySize = SibSp + Parch + 1
   Created. Range: 1 to 11
   Correlation with Survival: 0.0166

2. Title = Extracted from Name (Mr, Mrs, Miss, etc)
   Title distribution:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64

3. IsAlone = (FamilySize == 1)
   Alone: 537, Not Alone: 354

✅ Feature engineering complete!


---

## STEP 3: Prepare Data - Drop Unnecessary Columns

In [ ]:
print('\n' + '='*80)
print('DATA PREPARATION')
print('='*80)

# Drop columns not needed
df_clean = df_engineered.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Handle missing values
print('\nHandling missing values...')
print(f'Before: {df_clean.isnull().sum().sum()} missing values')

df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)

print(f'After: {df_clean.isnull().sum().sum()} missing values ✓')

# Separate features and target
X = df_clean.drop('Survived', axis=1)
y = df_clean['Survived']

print(f'\nFeatures: {X.shape[1]}')
print(f'Samples: {X.shape[0]}')
print(f'\nFeature columns:')
print(X.columns.tolist())

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'\nCategorical columns: {categorical_cols}')
print(f'Numerical columns: {numerical_cols}')


DATA PREPARATION

Handling missing values...
Before: 179 missing values
After: 0 missing values ✓

Features: 10
Samples: 891

Feature columns:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'Title', 'IsAlone']

Categorical columns: ['Sex', 'Embarked', 'Title']
Numerical columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']


---

## STEP 4: Train-Test Split

In [ ]:
print('\n' + '='*80)
print('TRAIN-TEST SPLIT')
print('='*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'\n✓ Stratified split maintains class balance')


TRAIN-TEST SPLIT

Training set: 712 samples
Test set: 179 samples

✓ Stratified split maintains class balance


---

## STEP 5: Build Production Pipeline - MANUAL APPROACH (Old Way)

In [ ]:
print('\n' + '='*80)
print('MANUAL APPROACH (OLD WAY - Prone to Errors)')
print('='*80)

from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Scale numerical
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Encode categorical
encoder = OneHotEncoder(sparse_output=False, drop='first')
X_train_encoded = encoder.fit_transform(X_train_scaled[categorical_cols])
X_test_encoded = encoder.transform(X_test_scaled[categorical_cols])

# Combine
X_train_final = np.concatenate([
    X_train_scaled[numerical_cols].values,
    X_train_encoded
], axis=1)

X_test_final = np.concatenate([
    X_test_scaled[numerical_cols].values,
    X_test_encoded
], axis=1)

# Train model
manual_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
manual_model.fit(X_train_final, y_train)

y_pred_manual = manual_model.predict(X_test_final)

manual_accuracy = accuracy_score(y_test, y_pred_manual)
manual_f1 = f1_score(y_test, y_pred_manual)

print(f'\n✅ Manual approach complete')
print(f'Accuracy: {manual_accuracy:.4f}')
print(f'F1-Score: {manual_f1:.4f}')

print(f'\n⚠️ PROBLEMS WITH MANUAL APPROACH:')
print(f'  • Easy to forget a step')
print(f'  • Different order = different results')
print(f'  • Risk of data leakage (fit on full data then split)')
print(f'  • Hard to reproduce')
print(f'  • Not scalable')


MANUAL APPROACH (OLD WAY - Prone to Errors)

✅ Manual approach complete
Accuracy: 0.8268
F1-Score: 0.7832

⚠️ PROBLEMS WITH MANUAL APPROACH:
  • Easy to forget a step
  • Different order = different results
  • Risk of data leakage (fit on full data then split)
  • Hard to reproduce
  • Not scalable


---

## STEP 6: Build Production Pipeline - CORRECT APPROACH (Pipeline)

In [ ]:
print('\n' + '='*80)
print('PRODUCTION PIPELINE APPROACH (CORRECT WAY)')
print('='*80)

# Step 1: Define preprocessing for numerical columns
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Step 2: Define preprocessing for categorical columns
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
])

# Step 3: Combine preprocessors using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Step 4: Create complete pipeline with preprocessing + model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

print('\nPipeline Structure:')
print(pipeline)

# Step 5: Fit pipeline (preprocessing happens automatically!)
print('\nFitting pipeline...')
pipeline.fit(X_train, y_train)
print('✓ Pipeline fitted')

# Step 6: Predict
y_pred_pipeline = pipeline.predict(X_test)

pipeline_accuracy = accuracy_score(y_test, y_pred_pipeline)
pipeline_f1 = f1_score(y_test, y_pred_pipeline)

print(f'\n✅ Pipeline approach complete')
print(f'Accuracy: {pipeline_accuracy:.4f}')
print(f'F1-Score: {pipeline_f1:.4f}')

print(f'\n✅ ADVANTAGES OF PIPELINE:')
print(f'  ✓ No data leakage (fit only on training data)')
print(f'  ✓ Consistent order (preprocessing always same)')
print(f'  ✓ Easy to reproduce')
print(f'  ✓ Scalable to production')
print(f'  ✓ Single object to deploy')


PRODUCTION PIPELINE APPROACH (CORRECT WAY)

Pipeline Structure:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                

---

## STEP 7: Compare Manual vs Pipeline Approach

In [ ]:
print('\n' + '='*80)
print('COMPARISON: MANUAL vs PIPELINE')
print('='*80)

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score', 'Same Results?', 'Data Leakage Risk?', 'Production Ready?'],
    'Manual Approach': [
        f'{manual_accuracy:.4f}',
        f'{manual_f1:.4f}',
        '❌ Risky',
        '⚠️ Yes (easy to make mistake)',
        '❌ No (too many steps)'
    ],
    'Pipeline Approach': [
        f'{pipeline_accuracy:.4f}',
        f'{pipeline_f1:.4f}',
        '✅ Guaranteed',
        '✅ No (built-in protection)',
        '✅ Yes (single object)'
    ]
})

print(f'\n{comparison.to_string(index=False)}')

print(f'\nPerformance Difference:')
acc_diff = abs(pipeline_accuracy - manual_accuracy)
print(f'Accuracy difference: {acc_diff:.6f} (essentially identical)')
print(f'→ Both approaches give same results, BUT pipeline is safer!')


COMPARISON: MANUAL vs PIPELINE

            Metric               Manual Approach          Pipeline Approach
          Accuracy                        0.8268                     0.8268
          F1-Score                        0.7832                     0.7832
     Same Results?                       ❌ Risky               ✅ Guaranteed
Data Leakage Risk? ⚠️ Yes (easy to make mistake) ✅ No (built-in protection)
 Production Ready?         ❌ No (too many steps)      ✅ Yes (single object)

Performance Difference:
Accuracy difference: 0.000000 (essentially identical)
→ Both approaches give same results, BUT pipeline is safer!


---

## STEP 8: Feature Engineering Impact Analysis

In [ ]:
print('\n' + '='*80)
print('FEATURE ENGINEERING IMPACT')
print('='*80)

print('\n🔧 NEW FEATURES CREATED:')
print('\n1. FamilySize:')
print(f'   - Definition: SibSp + Parch + 1')
print(f'   - Range: {df_engineered["FamilySize"].min()} to {df_engineered["FamilySize"].max()}')
print(f'   - Correlation with Survived: {df_engineered[["FamilySize", "Survived"]].corr().iloc[0, 1]:.4f}')
print(f'   - Insight: Family size affects survival')

print('\n2. Title:')
print(f'   - Definition: Extracted from Name (Mr, Mrs, Miss, Master, Other)')
print(f'   - Distribution:')
print(df_engineered['Title'].value_counts())
print(f'   - Insight: Title indicates gender/status → affects survival')

print('\n3. IsAlone:')
print(f'   - Definition: 1 if traveling alone, 0 otherwise')
print(f'   - Alone: {(df_engineered["IsAlone"]==1).sum()} passengers')
print(f'   - With family: {(df_engineered["IsAlone"]==0).sum()} passengers')
print(f'   - Insight: Traveling alone might affect survival chances')

print('\n📊 IMPACT ON MODEL:')
print(f'Original features (Task 3): 7 features')
print(f'Engineered features (Task 7): 10 features')
print(f'\nAccuracy improvement: {(pipeline_accuracy - manual_accuracy)*100:+.3f}%')
print(f'F1-Score improvement: {(pipeline_f1 - manual_f1)*100:+.3f}%')


FEATURE ENGINEERING IMPACT

🔧 NEW FEATURES CREATED:

1. FamilySize:
   - Definition: SibSp + Parch + 1
   - Range: 1 to 11
   - Correlation with Survived: 0.0166
   - Insight: Family size affects survival

2. Title:
   - Definition: Extracted from Name (Mr, Mrs, Miss, Master, Other)
   - Distribution:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64
   - Insight: Title indicates gender/status → affects survival

3. IsAlone:
   - Definition: 1 if traveling alone, 0 otherwise
   - Alone: 537 passengers
   - With family: 354 passengers
   - Insight: Traveling alone might affect survival chances

📊 IMPACT ON MODEL:
Original features (Task 3): 7 features
Engineered features (Task 7): 10 features

Accuracy improvement: +0.000%
F1-Score improvement: +0.000%


---

## STEP 9: Model Evaluation

In [ ]:
print('\n' + '='*80)
print('FINAL MODEL EVALUATION')
print('='*80)

precision = precision_score(y_test, y_pred_pipeline)
recall = recall_score(y_test, y_pred_pipeline)

print(f'\nAccuracy:  {pipeline_accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {pipeline_f1:.4f}')

print(f'\n{classification_report(y_test, y_pred_pipeline, target_names=["Not Survived", "Survived"])}')

# Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
print(f'\n5-Fold Cross-Validation F1-Scores:')
print(f'{cv_scores}')
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')


FINAL MODEL EVALUATION

Accuracy:  0.8268
Precision: 0.7568
Recall:    0.8116
F1-Score:  0.7832

              precision    recall  f1-score   support

Not Survived       0.88      0.84      0.86       110
    Survived       0.76      0.81      0.78        69

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179


5-Fold Cross-Validation F1-Scores:
[0.72413793 0.73873874 0.75630252 0.77477477 0.75229358]
Mean: 0.7492 (+/- 0.0170)


---

## STEP 10: Save Pipeline (Production Ready!)

In [ ]:
print('\n' + '='*80)
print('SAVING PIPELINE')
print('='*80)

# Method 1: Using joblib (RECOMMENDED for production)
joblib.dump(pipeline, 'titanic_pipeline_model.pkl')
print('\n✅ Saved with joblib: titanic_pipeline_model.pkl')

# Method 2: Using pickle (alternative)
with open('titanic_pipeline_backup.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print('✅ Saved with pickle: titanic_pipeline_backup.pkl')

print('\n💡 NOW YOU CAN:')
print('  • Share the .pkl file with team')
print('  • Deploy to production')
print('  • Load anytime with: joblib.load("titanic_pipeline_model.pkl")')
print('  • Use on new data: pipeline.predict(new_data)')
print('  • No data leakage risk')
print('  • Reproducible results')


SAVING PIPELINE

✅ Saved with joblib: titanic_pipeline_model.pkl
✅ Saved with pickle: titanic_pipeline_backup.pkl

💡 NOW YOU CAN:
  • Share the .pkl file with team
  • Deploy to production
  • Load anytime with: joblib.load("titanic_pipeline_model.pkl")
  • Use on new data: pipeline.predict(new_data)
  • No data leakage risk
  • Reproducible results


---

## STEP 11: Load and Test Saved Pipeline

In [ ]:
print('\n' + '='*80)
print('TESTING LOADED PIPELINE')
print('='*80)

# Load saved pipeline
loaded_pipeline = joblib.load('titanic_pipeline_model.pkl')
print('\n✅ Pipeline loaded from disk')

# Make predictions with loaded pipeline
y_pred_loaded = loaded_pipeline.predict(X_test)
accuracy_loaded = accuracy_score(y_test, y_pred_loaded)

print(f'\nAccuracy with loaded pipeline: {accuracy_loaded:.4f}')
print(f'Original accuracy: {pipeline_accuracy:.4f}')
print(f'Match: {accuracy_loaded == pipeline_accuracy} ✓')

print(f'\n🚀 Pipeline is production-ready!')
print(f'   • Saved successfully')
print(f'   • Loaded successfully')
print(f'   • Results identical')
print(f'   • Ready to deploy')


TESTING LOADED PIPELINE

✅ Pipeline loaded from disk

Accuracy with loaded pipeline: 0.8268
Original accuracy: 0.8268
Match: True ✓

🚀 Pipeline is production-ready!
   • Saved successfully
   • Loaded successfully
   • Results identical
   • Ready to deploy


---

## STEP 13: Final Summary

In [ ]:
print('\n' + '='*80)
print('🎉 TASK 7 COMPLETE - PRODUCTION ML PIPELINES')
print('='*80)

print(f'\n📊 DATASET')
print(f'Total Samples: {len(df)}')
print(f'Features Created: 3 new engineered features')
print(f'Total Features: {X.shape[1]}')

print(f'\n🏗️ PIPELINE ARCHITECTURE')
print(f'Numerical columns: {len(numerical_cols)} (StandardScaler)')
print(f'Categorical columns: {len(categorical_cols)} (OneHotEncoder)')
print(f'Model: Logistic Regression')
print(f'All combined in 1 reusable object')

print(f'\n✅ MODEL PERFORMANCE')
print(f'Accuracy: {pipeline_accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {pipeline_f1:.4f}')

print(f'\n✅ PIPELINE BENEFITS')
print(f'✓ No data leakage (fit only on training data)')
print(f'✓ Consistent results (same order always)')
print(f'✓ Reproducible (load pipeline, get same results)')
print(f'✓ Deployable (save with joblib, use anywhere)')
print(f'✓ Production-ready (single object to manage)')
print(f'✓ Scalable (easy to add more preprocessing steps)')

print(f'\n📁 SAVED ARTIFACTS')
print(f'Pipeline saved: titanic_pipeline_model.pkl')
print(f'Backup saved: titanic_pipeline_backup.pkl')
print(f'Load with: joblib.load("titanic_pipeline_model.pkl")')

print(f'\n✅ TASK 7 COMPLETE!')
print('='*80)


🎉 TASK 7 COMPLETE - PRODUCTION ML PIPELINES

📊 DATASET
Total Samples: 891
Features Created: 3 new engineered features
Total Features: 10

🏗️ PIPELINE ARCHITECTURE
Numerical columns: 7 (StandardScaler)
Categorical columns: 3 (OneHotEncoder)
Model: Logistic Regression
All combined in 1 reusable object

✅ MODEL PERFORMANCE
Accuracy: 0.8268
Precision: 0.7568
Recall: 0.8116
F1-Score: 0.7832

✅ PIPELINE BENEFITS
✓ No data leakage (fit only on training data)
✓ Consistent results (same order always)
✓ Reproducible (load pipeline, get same results)
✓ Deployable (save with joblib, use anywhere)
✓ Production-ready (single object to manage)
✓ Scalable (easy to add more preprocessing steps)

📁 SAVED ARTIFACTS
Pipeline saved: titanic_pipeline_model.pkl
Backup saved: titanic_pipeline_backup.pkl
Load with: joblib.load("titanic_pipeline_model.pkl")

✅ TASK 7 COMPLETE!
